[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/02_pytorch_core_building_blocks.ipynb)

# 02. PyTorch core building blocks

현대 모델의 가장 작은 공통 부품을 짧게 한 번씩 실행한다. 깊게 파지 않고 Linear, Conv, activation, normalization, residual, attention, loss, backward, optimizer step의 전체 흐름을 익힌다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


device: cuda
torch: 2.11.0+cu128


In [2]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Linear + activation


In [3]:
x = torch.tensor([[1.0, -1.0, 0.5, 2.0]], device=device)

linear = nn.Linear(4, 4, bias=True).to(device)
with torch.no_grad():
    linear.weight.copy_(torch.eye(4, device=device))
    linear.bias.zero_()

z = linear(x)
h = F.silu(z)

print("linear:", z)
print("SiLU:", h)


linear: tensor([[ 1.0000, -1.0000,  0.5000,  2.0000]], device='cuda:0',
       grad_fn=<AddmmBackward0>)
SiLU: tensor([[ 0.7311, -0.2689,  0.3112,  1.7616]], device='cuda:0',
       grad_fn=<SiluBackward0>)


## 2. Conv2d


In [4]:
img = torch.arange(25, dtype=torch.float32, device=device).reshape(1, 1, 5, 5)
conv = nn.Conv2d(1, 2, kernel_size=3, padding=1, bias=False).to(device)

y = conv(img)
print("input shape:", img.shape)
print("output shape:", y.shape)


input shape: torch.Size([1, 1, 5, 5])
output shape: torch.Size([1, 2, 5, 5])


## 3. Normalization + residual


In [5]:
x = torch.randn(2, 4, 8, device=device)
norm = nn.LayerNorm(8).to(device)
branch = nn.Linear(8, 8, bias=False).to(device)

y = x + branch(norm(x))

print("input norm:", x.norm().item())
print("output norm:", y.norm().item())


input norm: 8.015271186828613
output norm: 9.541211128234863


## 4. Minimal attention


In [6]:
B, T, D, H = 1, 4, 8, 2
x = torch.randn(B, T, D, device=device)

qkv = nn.Linear(D, 3 * D, bias=False).to(device)
q, k, v = qkv(x).chunk(3, dim=-1)

head_dim = D // H
q = q.view(B, T, H, head_dim).transpose(1, 2)
k = k.view(B, T, H, head_dim).transpose(1, 2)
v = v.view(B, T, H, head_dim).transpose(1, 2)

attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)
attn = attn.transpose(1, 2).reshape(B, T, D)

print("attention output shape:", attn.shape)


attention output shape: torch.Size([1, 4, 8])


## 5. Loss → backward → optimizer


In [7]:
model = nn.Linear(4, 2).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

x = torch.tensor([[1., 0., -1., 2.]], device=device)
target = torch.tensor([1], device=device)

logits = model(x)
loss = F.cross_entropy(logits, target)

optimizer.zero_grad()
loss.backward()

print("loss:", loss.item())
print("weight grad:\n", model.weight.grad)

optimizer.step()
print("updated weight:\n", model.weight)


loss: 1.5117383003234863
weight grad:
 tensor([[ 0.7795,  0.0000, -0.7795,  1.5589],
        [-0.7795, -0.0000,  0.7795, -1.5589]], device='cuda:0')
updated weight:
 Parameter containing:
tensor([[-0.0041,  0.3995, -0.1347,  0.3205],
        [-0.1536,  0.3200,  0.3588,  0.2076]], device='cuda:0',
       requires_grad=True)


## 6. Kernel view of the tiny training step


In [8]:
def tiny_train_step():
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = F.cross_entropy(logits, target)
    loss.backward()
    optimizer.step()
    return loss

_ = profile_call("tiny training step", tiny_train_step)



[tiny training step] top operators
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              aten::sum         0.39%      24.741us         0.54%      34.556us      34.556us      10.400us        22.63%      10.400us      10.400us           0 B     

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


## References and provenance

**[2.1] Residual learning**
- 출처: He et al., Deep Residual Learning for Image Recognition
- 이 노트북에서 가져온 부분: residual branch의 최소 구조

**[2.2] Scaled dot-product attention**
- 출처: Vaswani et al., Attention Is All You Need
- 이 노트북에서 가져온 부분: Q/K/V 기반 attention의 최소 구조
